# Simulação de um modelo de predação: lobos, ovelhas e grama — V9 comentada e estável

Este notebook foi construído com apoio do ChatGPT para uso didático no SimServ.

A simulação é implementada em Python, inspirada no modelo original **Wolf Sheep Predation** da biblioteca de exemplos do NetLogo e na abordagem apresentada em Wilensky, U. & Rand, W. (2015), *Introduction to Agent-Based Modeling: Modeling Natural, Social and Engineered Complex Systems with NetLogo*. Cambridge, MA: MIT Press.

A versão V9 preserva a arquitetura que funcionou na V5: o painel principal é desenhado em **SVG/HTML persistente** dentro de `ipywidgets.HTML`. O objetivo é evitar a instabilidade observada com `matplotlib`, `plt.show()` e `widgets.Output()` em alguns ambientes JupyterLab.

## Fluxo didático esperado

1. Opcionalmente, rode a Célula 1 para verificar se widgets HTML persistem no JupyterLab.
2. Rode as células 2, 3 e 4 para carregar bibliotecas, modelo e funções gráficas.
3. Rode a Célula 5 para abrir o painel principal.
4. Ajuste os sliders.
5. Pressione **SETUP** para criar a população inicial.
6. Pressione **1 TICK**, **GO** ou **ANIMAR** para observar a dinâmica.
7. Compare os resultados mudando os parâmetros e repetindo **SETUP**.

## Regras principais do modelo

- A paisagem é formada por *patches* com uma quantidade de grama (`grass-amount`).
- Ovelhas e lobos possuem energia.
- Agentes se movem e perdem energia pelo custo de movimento.
- Ovelhas comem grama e ganham energia.
- Lobos comem ovelhas e ganham energia.
- Agentes com energia maior que 200 podem se reproduzir.
- Agentes com energia menor que 0 morrem.
- A grama cresce novamente a cada *tick*.

## Parâmetros controlados no painel

| Parâmetro | Intervalo | Significado |
|---|---:|---|
| `NUMBER-OF-SHEEP` | 0 a 300 | Número inicial de ovelhas. |
| `NUMBER-OF-WOLVES` | 0 a 150 | Número inicial de lobos. |
| `MOVEMENT-COST` | 0 a 10 | Energia perdida por movimento. |
| `GRASS-REGROWTH-RATE` | 0 a 2 | Velocidade de crescimento da grama. |
| `ENERGY-GAIN-FROM-GRASS` | 0 a 20 | Energia ganha por ovelha ao comer grama. |
| `ENERGY-GAIN-FROM-SHEEP` | 0 a 80 | Energia ganha por lobo ao comer ovelha. |
| `WORLD-SIZE` | 21 a 81 | Tamanho lateral da paisagem quadrada. |
| `SEED` | 1 a 999 | Semente aleatória para repetir experimentos. |
| `TICKS POR GO` | 1 a 300 | Quantos passos são executados pelo botão **GO**. |
| `TICKS DA ANIMAÇÃO` | 10 a 500 | Quantos passos podem ser executados pelo botão **ANIMAR**. |
| `PAUSA DA ANIMAÇÃO` | 0 a 0,5 s | Intervalo visual entre ticks durante a animação. |

## Observação técnica importante

Em alguns frontends JupyterLab, o primeiro widget de uma sessão pode aparecer de relance enquanto o gerenciador JavaScript de widgets é carregado. A Célula 5 desta V9 foi ajustada para inicializar os valores **antes** de exibir o painel e deixar o painel como **última saída da célula**, o que reduz esse problema.


## Teste opcional de persistência HTML + widgets

Esta célula **não faz parte da simulação**. Ela foi mantida apenas como diagnóstico rápido do ambiente JupyterLab.

Rode esta célula somente se quiser verificar se `ipywidgets.HTML`, sliders e botões permanecem visíveis na saída da célula. Como esse teste já funcionou no SimServ, ele não precisa ser repetido toda vez que o notebook for usado em aula.


In [ ]:
# ============================================================
# CÉLULA 1 — Teste mínimo de persistência HTML + widgets
# ============================================================
# Este teste é opcional. Ele não faz parte do modelo de predação.
# Sua função é apenas verificar se o JupyterLab consegue manter
# ipywidgets.HTML, sliders e botões como saída persistente da célula.
#
# Na V9, o painel é deixado como última expressão da célula, sem
# comandos posteriores de atualização. Isso é mais estável no JupyterLab.

import ipywidgets as widgets
from IPython.display import display

_teste_contador = widgets.IntText(value=0, description='Cliques:', disabled=True)
_teste_botao = widgets.Button(description='Clique para testar', button_style='success')
_teste_slider = widgets.IntSlider(value=5, min=0, max=10, description='Slider:')
_teste_html = widgets.HTML(value=r'''
<div style="border:2px solid #4a7; padding:10px; max-width:620px; background:#f8fff8">
<b>Teste visual persistente:</b> se este quadro, o slider e o botão permanecem visíveis,
a camada básica de widgets está funcionando.<br>
<svg width="560" height="90" viewBox="0 0 560 90" xmlns="http://www.w3.org/2000/svg">
  <rect x="10" y="10" width="540" height="70" rx="8" fill="#edf7ed" stroke="#377"/>
  <circle cx="80" cy="45" r="18" fill="white" stroke="black"/>
  <text x="73" y="51" font-size="17" font-family="sans-serif">S</text>
  <circle cx="170" cy="45" r="18" fill="#8a5a2b" stroke="black"/>
  <text x="163" y="51" font-size="17" font-family="sans-serif" fill="white">W</text>
  <text x="225" y="50" font-size="17" fill="#164" font-family="sans-serif">HTML/SVG persistente no notebook</text>
</svg>
</div>
''')


def _teste_click(_):
    # Callback simples: cada clique incrementa o contador.
    _teste_contador.value += 1


_teste_botao.on_click(_teste_click)

_teste_panel = widgets.VBox([
    widgets.HTML('<h3>Célula 1 — teste opcional de persistência</h3>'),
    _teste_html,
    widgets.HBox([_teste_slider, _teste_botao, _teste_contador])
])

# Última expressão da célula: o JupyterLab renderiza este widget como saída.
_teste_panel


## Bibliotecas usadas pelo modelo

Esta célula prepara o ambiente de software para a simulação.

As mensagens de saída são úteis para o aluno confirmar que as bibliotecas foram importadas corretamente e que pode continuar para as próximas células.

Observação: o painel principal usa **SVG/HTML**, não `matplotlib`. A biblioteca `numpy` é usada principalmente para representar a matriz de grama do ambiente.


In [ ]:
# ============================================================
# CÉLULA 2 — Bibliotecas usadas pelo modelo
# ============================================================
# O painel principal usa SVG/HTML, não matplotlib.
# numpy é usado apenas para representar a matriz de grama da paisagem.
#
# Comentário didático:
# Esta célula prepara o ambiente de software para a simulação.
# Se a mensagem final aparecer, o aluno pode seguir para a definição do modelo.

import math
import random
import csv
import asyncio
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, FileLink

HTML(f'''
<div style="border:1px solid #7a7; background:#f7fff7; padding:8px; max-width:720px">
<b>Bibliotecas carregadas com sucesso.</b><br>
Python preparado para executar o modelo ABM lobos-ovelhas.<br>
<code>numpy</code>: {np.__version__} &nbsp; | &nbsp;
<code>ipywidgets</code>: {widgets.__version__}
</div>
''')


## Núcleo do modelo ABM lobos–ovelhas

Esta célula define o modelo baseado em agentes.

Leia o código procurando reconhecer os blocos principais:

- `@dataclass Agent`: estrutura simples para representar cada agente;
- `class WolfSheepModel`: classe que guarda o estado completo da simulação;
- `__init__`: cria o ambiente, a grama, as ovelhas e os lobos;
- `_new_agent`: cria um novo agente;
- `wiggle_and_move`: implementa o movimento e o custo de energia;
- `sheep_step`: regras das ovelhas;
- `wolf_step`: regras dos lobos;
- `reproduce`: reprodução quando há energia suficiente;
- `step`: executa um tick completo da simulação;
- `record`: registra os dados para os gráficos;
- `save_csv`: salva o histórico da simulação.

A intenção não é que o aluno memorize o código, mas que consiga localizar onde estão as regras do modelo.


In [ ]:
# ============================================================
# CÉLULA 3 — Núcleo do modelo ABM lobos-ovelhas
# ============================================================
# Implementação Python inspirada diretamente no NetLogo anexado.
#
# Comentário didático dos blocos principais:
# - Agent representa uma ovelha ou lobo com posição, direção e energia.
# - WolfSheepModel guarda o ambiente, as populações e o histórico.
# - step() executa um tick: grama cresce, ovelhas agem e lobos agem.
# - record() grava os dados usados nos gráficos.

@dataclass
class Agent:
    """Representa um agente individual da simulação.

    No NetLogo, ovelhas e lobos são turtles de breeds diferentes.
    Aqui usamos uma mesma estrutura Agent, com o campo kind indicando
    se o agente é 'sheep' ou 'wolf'.
    """
    kind: str       # 'sheep' ou 'wolf'
    x: float
    y: float
    heading: float
    energy: float = 100.0
    alive: bool = True


class WolfSheepModel:
    """Núcleo do modelo ABM lobos-ovelhas.

    Esta classe guarda simultaneamente:
    - o ambiente, representado pela matriz grass;
    - as populações de ovelhas e lobos;
    - os parâmetros experimentais;
    - o histórico usado para construir os gráficos.
    """

    def __init__(
        self,
        number_of_sheep: int = 100,
        number_of_wolves: int = 30,
        world_size: int = 51,
        movement_cost: float = 1.0,
        grass_regrowth_rate: float = 0.10,
        energy_gain_from_grass: float = 4.0,
        energy_gain_from_sheep: float = 20.0,
        seed: int = 7,
        max_agents: int = 5000,
    ):
        self.number_of_sheep = int(number_of_sheep)
        self.number_of_wolves = int(number_of_wolves)
        self.world_size = int(world_size)
        self.movement_cost = float(movement_cost)
        self.grass_regrowth_rate = float(grass_regrowth_rate)
        self.energy_gain_from_grass = float(energy_gain_from_grass)
        self.energy_gain_from_sheep = float(energy_gain_from_sheep)
        self.seed = int(seed)
        self.max_agents = int(max_agents)

        self.rng = random.Random(self.seed)
        self.np_rng = np.random.default_rng(self.seed)
        self.tick = 0
        self.warning = ''

        self.grass = self.np_rng.random((self.world_size, self.world_size)) * 10.0
        self.sheep: List[Agent] = [self._new_agent('sheep') for _ in range(self.number_of_sheep)]
        self.wolves: List[Agent] = [self._new_agent('wolf') for _ in range(self.number_of_wolves)]
        self.history: List[Dict[str, float]] = []
        self.record()

    def _new_agent(self, kind: str, x=None, y=None) -> Agent:
        return Agent(
            kind=kind,
            x=self.rng.random() * self.world_size if x is None else float(x) % self.world_size,
            y=self.rng.random() * self.world_size if y is None else float(y) % self.world_size,
            heading=self.rng.random() * 2.0 * math.pi,
            energy=100.0,
            alive=True,
        )

    def patch_index(self, agent: Agent) -> Tuple[int, int]:
        # x corresponde à coluna; y corresponde à linha.
        return int(agent.x) % self.world_size, int(agent.y) % self.world_size

    def wiggle_and_move(self, agent: Agent):
        # NetLogo: rt random 90; lt random 90; forward 1; energy -= movement-cost
        agent.heading += math.radians(self.rng.uniform(0, 90))
        agent.heading -= math.radians(self.rng.uniform(0, 90))
        agent.x = (agent.x + math.cos(agent.heading)) % self.world_size
        agent.y = (agent.y + math.sin(agent.heading)) % self.world_size
        agent.energy -= self.movement_cost

    def eat_grass(self, sheep: Agent):
        px, py = self.patch_index(sheep)
        if self.grass[py, px] >= self.energy_gain_from_grass:
            sheep.energy += self.energy_gain_from_grass
            self.grass[py, px] -= self.energy_gain_from_grass

    def eat_sheep(self, wolf: Agent):
        wx, wy = self.patch_index(wolf)
        candidates = [s for s in self.sheep if s.alive and self.patch_index(s) == (wx, wy)]
        if candidates:
            target = self.rng.choice(candidates)
            target.alive = False
            wolf.energy += self.energy_gain_from_sheep

    def reproduce_if_possible(self, agent: Agent, newborns: List[Agent]):
        if agent.energy > 200 and self.total_agents() + len(newborns) < self.max_agents:
            agent.energy -= 100
            child = self._new_agent(agent.kind, x=agent.x, y=agent.y)
            child.heading = agent.heading
            child.energy = 100.0
            newborns.append(child)
        elif agent.energy > 200 and self.total_agents() + len(newborns) >= self.max_agents:
            self.warning = f'Limite de segurança de {self.max_agents} agentes atingido; reprodução suspensa.'

    def total_agents(self) -> int:
        return len(self.sheep) + len(self.wolves)

    def step(self):
        # Um tick completo da simulação:
        # 1. grama cresce;
        # 2. ovelhas se movem, comem, reproduzem ou morrem;
        # 3. lobos se movem, comem ovelhas, reproduzem ou morrem;
        # 4. o histórico é atualizado.
        if self.total_agents() == 0:
            self.warning = 'Não há agentes vivos. Rode SETUP para reiniciar.'
            return

        newborn_sheep: List[Agent] = []
        newborn_wolves: List[Agent] = []
        agents = self.sheep + self.wolves
        self.rng.shuffle(agents)

        for agent in agents:
            if not agent.alive:
                continue

            self.wiggle_and_move(agent)

            # NetLogo faz check-if-dead logo após mover.
            if agent.energy < 0:
                agent.alive = False
                continue

            if agent.kind == 'sheep':
                self.eat_grass(agent)
                self.reproduce_if_possible(agent, newborn_sheep)
            else:
                self.eat_sheep(agent)
                self.reproduce_if_possible(agent, newborn_wolves)

        self.sheep = [s for s in self.sheep if s.alive] + newborn_sheep
        self.wolves = [w for w in self.wolves if w.alive] + newborn_wolves
        self.grass = np.minimum(10.0, self.grass + self.grass_regrowth_rate)
        self.tick += 1
        self.record()

    def run(self, n_ticks: int):
        for _ in range(int(n_ticks)):
            if self.total_agents() == 0:
                break
            self.step()

    def counts(self) -> Tuple[int, int]:
        return len(self.sheep), len(self.wolves)

    def mean_energy(self, agents: List[Agent]) -> float:
        return float(np.mean([a.energy for a in agents])) if agents else 0.0

    def record(self):
        # Guarda uma fotografia numérica do sistema no tick atual.
        # Esses dados são usados pelos gráficos e pelo arquivo CSV.
        n_sheep, n_wolves = self.counts()
        self.history.append({
            'tick': self.tick,
            'sheep': n_sheep,
            'wolves': n_wolves,
            'wolves_x10': n_wolves * 10,
            'grass_scaled': float(self.grass.sum() / 50.0),
            'grass_total': float(self.grass.sum()),
            'mean_energy_sheep': self.mean_energy(self.sheep),
            'mean_energy_wolves': self.mean_energy(self.wolves),
        })

    def save_csv(self, filename: str):
        if not self.history:
            return
        fields = list(self.history[0].keys())
        with open(filename, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fields)
            writer.writeheader()
            writer.writerows(self.history)


## Desenho persistente em SVG/HTML

Esta célula define as funções gráficas usadas pelo painel.

Ela **não deve mostrar figura sozinha**. Seu papel é carregar funções que serão chamadas pela célula do painel principal.

A visualização é feita com strings HTML/SVG:

- `svg_landscape`: desenha a paisagem, a grama, as ovelhas e os lobos;
- `svg_timeseries`: desenha séries temporais como segmentos SVG;
- `status_html`: mostra o estado numérico atual;
- `panel_html`: combina paisagem, gráficos e status em um painel;
- `html_report`: gera um relatório visual estático.

Essa decisão torna a visualização mais robusta no JupyterLab do SimServ do que depender de figuras temporárias do `matplotlib`.


In [ ]:
# ============================================================
# CÉLULA 4 — Desenho persistente em SVG/HTML
# ============================================================
# Não usa matplotlib. Os resultados são strings HTML/SVG.
#
# Comentário didático:
# Esta célula não deve produzir figura diretamente. Ela define funções
# chamadas pelo painel principal para desenhar paisagem, gráficos e status.


def _clamp(v, lo, hi):
    return max(lo, min(hi, v))


def grass_color(amount: float) -> str:
    # Mapeia 0..10 para tons de marrom/verde.
    a = _clamp(amount / 10.0, 0.0, 1.0)
    r = int(215 - 150 * a)
    g = int(170 + 70 * a)
    b = int(95 - 55 * a)
    return f'rgb({r},{g},{b})'


def svg_landscape(model: WolfSheepModel, width: int = 620) -> str:
    # Produz a paisagem como SVG: cada patch vira um pequeno retângulo
    # colorido conforme a quantidade de grama. Ovelhas e lobos são
    # desenhados por cima como círculos.
    n = model.world_size
    cell = width / n
    h = width
    parts = [
        f'<svg width="{width}" height="{h}" viewBox="0 0 {width} {h}" xmlns="http://www.w3.org/2000/svg">',
        f'<rect x="0" y="0" width="{width}" height="{h}" fill="#eef"/>'
    ]

    # Patches de grama.
    for y in range(n):
        yy = y * cell
        for x in range(n):
            xx = x * cell
            parts.append(
                f'<rect x="{xx:.2f}" y="{yy:.2f}" width="{cell+0.2:.2f}" height="{cell+0.2:.2f}" fill="{grass_color(model.grass[y, x])}"/>'
            )

    # Agentes. Amostragem apenas se a população explodir.
    max_sheep_draw = 900
    max_wolves_draw = 500
    sheep_to_draw = model.sheep[:max_sheep_draw]
    wolves_to_draw = model.wolves[:max_wolves_draw]
    r_sheep = max(2.0, min(5.0, cell * 0.38))
    r_wolf = max(2.5, min(6.0, cell * 0.48))

    for s in sheep_to_draw:
        cx = s.x * cell
        cy = s.y * cell
        parts.append(f'<circle cx="{cx:.2f}" cy="{cy:.2f}" r="{r_sheep:.2f}" fill="white" stroke="#333" stroke-width="0.7"/>')

    for w in wolves_to_draw:
        cx = w.x * cell
        cy = w.y * cell
        parts.append(f'<circle cx="{cx:.2f}" cy="{cy:.2f}" r="{r_wolf:.2f}" fill="#7a4a22" stroke="#111" stroke-width="0.9"/>')

    # Borda e legenda mínima.
    parts.append(f'<rect x="0" y="0" width="{width}" height="{h}" fill="none" stroke="#222" stroke-width="1.2"/>')
    parts.append('</svg>')
    note = ''
    if len(model.sheep) > max_sheep_draw or len(model.wolves) > max_wolves_draw:
        note = '<div style="font-size:12px; color:#934">Obs.: população muito grande; alguns agentes foram omitidos apenas no desenho.</div>'
    return '<div>' + ''.join(parts) + note + '</div>'


def _series_points(values, width, height, pad_left, pad_right, pad_top, pad_bottom, ymin, ymax):
    if len(values) == 1:
        x = pad_left
        y = height - pad_bottom - (values[0] - ymin) / (ymax - ymin) * (height - pad_top - pad_bottom)
        return [(x, y)]
    points = []
    usable_w = width - pad_left - pad_right
    usable_h = height - pad_top - pad_bottom
    for i, v in enumerate(values):
        x = pad_left + usable_w * i / (len(values) - 1)
        y = height - pad_bottom - (v - ymin) / (ymax - ymin) * usable_h
        points.append((x, y))
    return points


def svg_line_chart(title: str, series: List[Tuple[str, List[float], str]], width: int = 760, height: int = 260, ylabel: str = '') -> str:
    pad_left, pad_right, pad_top, pad_bottom = 58, 22, 38, 42
    all_values = [v for _, vals, _ in series for v in vals]
    if not all_values:
        all_values = [0]
    ymin = 0.0
    ymax = max(1.0, max(all_values) * 1.12)
    if ymax == ymin:
        ymax = ymin + 1.0

    parts = [f'<svg width="{width}" height="{height}" viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg">']
    parts.append(f'<rect x="0" y="0" width="{width}" height="{height}" fill="white" stroke="#ddd"/>')
    parts.append(f'<text x="{width/2:.0f}" y="22" font-size="17" text-anchor="middle" font-family="sans-serif">{title}</text>')

    # Grades horizontais.
    for frac in [0, 0.25, 0.5, 0.75, 1.0]:
        y = height - pad_bottom - frac * (height - pad_top - pad_bottom)
        val = ymin + frac * (ymax - ymin)
        parts.append(f'<line x1="{pad_left}" y1="{y:.1f}" x2="{width-pad_right}" y2="{y:.1f}" stroke="#eee"/>')
        parts.append(f'<text x="{pad_left-8}" y="{y+4:.1f}" font-size="11" text-anchor="end" font-family="sans-serif">{val:.0f}</text>')

    # Eixos.
    parts.append(f'<line x1="{pad_left}" y1="{height-pad_bottom}" x2="{width-pad_right}" y2="{height-pad_bottom}" stroke="#555"/>')
    parts.append(f'<line x1="{pad_left}" y1="{pad_top}" x2="{pad_left}" y2="{height-pad_bottom}" stroke="#555"/>')
    if ylabel:
        parts.append(f'<text x="16" y="{height/2:.0f}" font-size="12" text-anchor="middle" transform="rotate(-90 16 {height/2:.0f})" font-family="sans-serif">{ylabel}</text>')

    # Curvas e pontos.
    legend_x = pad_left + 8
    legend_y = height - 14
    for idx, (label, vals, color) in enumerate(series):
        pts = _series_points(vals, width, height, pad_left, pad_right, pad_top, pad_bottom, ymin, ymax)
        if len(pts) >= 2:
            points_str = ' '.join(f'{x:.1f},{y:.1f}' for x, y in pts)
            parts.append(f'<polyline points="{points_str}" fill="none" stroke="{color}" stroke-width="2.4"/>')
        if pts:
            x, y = pts[-1]
            parts.append(f'<circle cx="{x:.1f}" cy="{y:.1f}" r="3.8" fill="{color}"/>')
        lx = legend_x + idx * 170
        parts.append(f'<rect x="{lx}" y="{legend_y-10}" width="16" height="4" fill="{color}"/>')
        parts.append(f'<text x="{lx+22}" y="{legend_y-5}" font-size="12" font-family="sans-serif">{label}</text>')

    n_ticks = max(len(vals) for _, vals, _ in series) - 1 if series else 0
    parts.append(f'<text x="{width/2:.0f}" y="{height-6}" font-size="12" text-anchor="middle" font-family="sans-serif">ticks: 0 a {n_ticks}</text>')
    parts.append('</svg>')
    return ''.join(parts)


def html_population_chart(model: WolfSheepModel) -> str:
    h = model.history
    return svg_line_chart(
        'Populações e grama — escala semelhante ao NetLogo',
        [
            ('ovelhas', [row['sheep'] for row in h], '#222222'),
            ('lobos x 10', [row['wolves_x10'] for row in h], '#8a4b1f'),
            ('grama / 50', [row['grass_scaled'] for row in h], '#168a3a'),
        ],
        ylabel='valor escalado'
    )


def html_energy_chart(model: WolfSheepModel) -> str:
    h = model.history
    return svg_line_chart(
        'Energia média dos agentes',
        [
            ('ovelhas', [row['mean_energy_sheep'] for row in h], '#333333'),
            ('lobos', [row['mean_energy_wolves'] for row in h], '#8a4b1f'),
        ],
        ylabel='energia média'
    )


def status_html(model: WolfSheepModel, extra: str = '') -> str:
    n_sheep, n_wolves = model.counts()
    last = model.history[-1]
    warning = f'<div style="color:#a33"><b>Aviso:</b> {model.warning}</div>' if model.warning else ''
    extra_html = f'<div style="margin-top:4px; color:#246"><b>Mensagem:</b> {extra}</div>' if extra else ''
    return f'''
    <div style="border:1px solid #bbb; background:#fafafa; padding:10px; font-family:sans-serif; max-width:760px">
      <b>Estado do modelo</b><br>
      tick = <b>{model.tick}</b> &nbsp; | &nbsp;
      ovelhas = <b>{n_sheep}</b> &nbsp; | &nbsp;
      lobos = <b>{n_wolves}</b> &nbsp; | &nbsp;
      grama total = <b>{last['grass_total']:.1f}</b><br>
      energia média das ovelhas = <b>{last['mean_energy_sheep']:.2f}</b> &nbsp; | &nbsp;
      energia média dos lobos = <b>{last['mean_energy_wolves']:.2f}</b>
      {extra_html}
      {warning}
    </div>
    '''


def html_report(model: WolfSheepModel) -> str:
    return f'''
    <div style="font-family:sans-serif">
      <h2>Relatório visual — Wolf Sheep Predation</h2>
      {status_html(model, 'relatório gerado pelo fallback sem widgets')}
      <h3>1. Paisagem</h3>
      {svg_landscape(model, width=620)}
      <h3>2. Populações e grama</h3>
      {html_population_chart(model)}
      <h3>3. Energia média</h3>
      {html_energy_chart(model)}
    </div>
    '''


## Painel principal de experimentação

Esta é a célula central para uso em aula.

Ela cria um painel persistente no estilo NetLogo, com sliders, botões e gráficos. O aluno deve:

1. ajustar os parâmetros;
2. pressionar **SETUP**;
3. observar a condição inicial;
4. usar **1 TICK**, **GO** ou **ANIMAR**;
5. comparar os gráficos após mudar os parâmetros.

A célula usa a mesma arquitetura da V5, que funcionou no ambiente JupyterLab do SimServ: `ipywidgets.HTML` com SVG/HTML persistente. Por estabilidade, a lógica do painel foi preservada.


In [ ]:
# ============================================================
# CÉLULA 5 — Painel principal estilo NetLogo, persistente em HTML/SVG
# ============================================================
# Este painel deve mostrar sliders, botões e gráficos persistentes.
# Ele não usa matplotlib nem widgets.Output.
#
# Ajuste V9 para estabilidade no SimServ:
# - os widgets HTML são preenchidos ANTES da exibição;
# - o painel é a última expressão da célula;
# - não há comando posterior a ele que possa substituir a saída visual.

# ----------------------------
# Sliders do modelo
# ----------------------------
style = {'description_width': '190px'}
layout = widgets.Layout(width='560px')

number_of_sheep_slider = widgets.IntSlider(value=100, min=0, max=300, step=5, description='NUMBER-OF-SHEEP', continuous_update=False, style=style, layout=layout)
number_of_wolves_slider = widgets.IntSlider(value=30, min=0, max=150, step=5, description='NUMBER-OF-WOLVES', continuous_update=False, style=style, layout=layout)
movement_cost_slider = widgets.FloatSlider(value=1.0, min=0.0, max=10.0, step=0.25, description='MOVEMENT-COST', continuous_update=False, style=style, layout=layout)
grass_regrowth_slider = widgets.FloatSlider(value=0.10, min=0.0, max=2.0, step=0.05, description='GRASS-REGROWTH-RATE', continuous_update=False, style=style, layout=layout)
energy_grass_slider = widgets.FloatSlider(value=4.0, min=0.0, max=20.0, step=0.5, description='ENERGY-GAIN-FROM-GRASS', continuous_update=False, style=style, layout=layout)
energy_sheep_slider = widgets.FloatSlider(value=20.0, min=0.0, max=80.0, step=1.0, description='ENERGY-GAIN-FROM-SHEEP', continuous_update=False, style=style, layout=layout)
world_size_slider = widgets.IntSlider(value=51, min=21, max=81, step=10, description='WORLD-SIZE', continuous_update=False, style=style, layout=layout)
seed_slider = widgets.IntSlider(value=7, min=1, max=999, step=1, description='SEED', continuous_update=False, style=style, layout=layout)
go_ticks_slider = widgets.IntSlider(value=50, min=1, max=300, step=1, description='TICKS POR GO', continuous_update=False, style=style, layout=layout)
animation_ticks_slider = widgets.IntSlider(value=100, min=10, max=500, step=10, description='TICKS DA ANIMACAO', continuous_update=False, style=style, layout=layout)
animation_delay_slider = widgets.FloatSlider(value=0.08, min=0.00, max=0.50, step=0.02, description='PAUSA DA ANIMACAO', continuous_update=False, style=style, layout=layout)

# ----------------------------
# Botões
# ----------------------------
setup_button = widgets.Button(description='SETUP', button_style='success', icon='refresh', layout=widgets.Layout(width='145px'))
one_tick_button = widgets.Button(description='1 TICK', button_style='info', icon='step-forward', layout=widgets.Layout(width='145px'))
go_button = widgets.Button(description='GO', button_style='primary', icon='play', layout=widgets.Layout(width='145px'))
animate_button = widgets.Button(description='ANIMAR', button_style='warning', icon='play-circle', layout=widgets.Layout(width='145px'))
stop_button = widgets.Button(description='PARAR', button_style='danger', icon='stop', layout=widgets.Layout(width='145px'))
save_button = widgets.Button(description='SALVAR CSV', button_style='', icon='save', layout=widgets.Layout(width='145px'))

# ----------------------------
# Áreas persistentes de visualização
# ----------------------------
status_view = widgets.HTML(value='')
landscape_view = widgets.HTML(value='')
population_view = widgets.HTML(value='')
energy_view = widgets.HTML(value='')

state = {'model': None, 'running': False, 'last_csv': ''}


def model_from_sliders() -> WolfSheepModel:
    return WolfSheepModel(
        number_of_sheep=number_of_sheep_slider.value,
        number_of_wolves=number_of_wolves_slider.value,
        world_size=world_size_slider.value,
        movement_cost=movement_cost_slider.value,
        grass_regrowth_rate=grass_regrowth_slider.value,
        energy_gain_from_grass=energy_grass_slider.value,
        energy_gain_from_sheep=energy_sheep_slider.value,
        seed=seed_slider.value,
    )


def update_panel(message: str = ''):
    model = state['model']
    if model is None:
        return
    status_view.value = status_html(model, message)
    landscape_view.value = svg_landscape(model, width=620)
    population_view.value = html_population_chart(model)
    energy_view.value = html_energy_chart(model)


def setup_model(_=None):
    state['model'] = model_from_sliders()
    state['running'] = False
    update_panel('SETUP executado. Populacao inicial criada.')


def one_tick(_=None):
    if state['model'] is None:
        setup_model()
    state['model'].step()
    update_panel('Avancou 1 tick.')


def go_ticks(_=None):
    if state['model'] is None:
        setup_model()
    n = go_ticks_slider.value
    state['model'].run(n)
    update_panel(f'GO executado: avancou {n} ticks.')


async def animate_ticks_async():
    if state['model'] is None:
        setup_model()
    if state['running']:
        return
    state['running'] = True
    n = animation_ticks_slider.value
    delay = animation_delay_slider.value
    for i in range(n):
        if not state['running']:
            update_panel('Animacao interrompida pelo botao PARAR.')
            return
        if state['model'].total_agents() == 0:
            update_panel('Animacao encerrada: nao ha agentes vivos.')
            state['running'] = False
            return
        state['model'].step()
        # Atualizar a cada tick torna a experiencia mais parecida com o NetLogo.
        update_panel(f'Animacao em andamento: tick {state["model"].tick}.')
        await asyncio.sleep(delay)
    state['running'] = False
    update_panel(f'Animacao concluida: {n} ticks executados.')


def start_animation(_=None):
    asyncio.create_task(animate_ticks_async())


def stop_animation(_=None):
    state['running'] = False


def save_csv(_=None):
    if state['model'] is None:
        setup_model()
    fname = f'wolf_sheep_historico_v5_seed{seed_slider.value}_tick{state["model"].tick}.csv'
    state['model'].save_csv(fname)
    state['last_csv'] = fname
    update_panel(f'Historico salvo em {fname}.')

setup_button.on_click(setup_model)
one_tick_button.on_click(one_tick)
go_button.on_click(go_ticks)
animate_button.on_click(start_animation)
stop_button.on_click(stop_animation)
save_button.on_click(save_csv)

controls = widgets.VBox([
    widgets.HTML('<h3>Controles do modelo</h3>'),
    number_of_sheep_slider,
    number_of_wolves_slider,
    movement_cost_slider,
    grass_regrowth_slider,
    energy_grass_slider,
    energy_sheep_slider,
    world_size_slider,
    seed_slider,
    widgets.HTML('<hr><b>Execucao</b>'),
    go_ticks_slider,
    animation_ticks_slider,
    animation_delay_slider,
    widgets.HBox([setup_button, one_tick_button, go_button]),
    widgets.HBox([animate_button, stop_button, save_button]),
    widgets.HTML('''
    <div style="font-size:13px; color:#444; margin-top:8px">
    Fluxo sugerido: ajuste os parametros -> pressione <b>SETUP</b> -> pressione <b>1 TICK</b>, <b>GO</b> ou <b>ANIMAR</b>.
    </div>
    '''),
], layout=widgets.Layout(width='590px'))

results = widgets.VBox([
    widgets.HTML('<h3>Resultados persistentes em SVG/HTML</h3>'),
    status_view,
    widgets.HTML('<h4>1. Paisagem: grama, ovelhas e lobos</h4>'),
    landscape_view,
    widgets.HTML('<h4>2. Populacoes e grama</h4><div style="font-size:13px">Curvas: ovelhas, lobos x 10 e grama / 50.</div>'),
    population_view,
    widgets.HTML('<h4>3. Energia media</h4>'),
    energy_view,
], layout=widgets.Layout(width='820px'))

panel = widgets.HBox([controls, results])

# Inicializamos o modelo e preenchemos os HTMLs antes de exibir o painel.
# Isso reduz o efeito observado no JupyterLab em que o painel aparece
# de relance e desaparece quando o frontend ainda está inicializando widgets.
setup_model()

# Mantemos uma referência global ao painel para evitar coleta prematura
# e para facilitar depuração em aula, se necessário.
wolf_sheep_panel = panel

# Última expressão da célula: esta é a saída visual principal.
wolf_sheep_panel


## Interpretação didática dos gráficos

No painel principal, os gráficos ajudam a acompanhar a evolução do sistema:

- **Paisagem**: mostra patches de grama e a posição instantânea dos agentes.
- **Populações e grama**: mostra a evolução temporal das ovelhas, dos lobos e da grama. Algumas grandezas podem estar escaladas para caber no mesmo gráfico.
- **Energia média**: mostra a energia média das populações sobreviventes.

Quando uma população desaparece, o gráfico deixa de registrar energia média para aquela população. Isso é uma indicação importante de extinção no experimento.

## Perguntas orientadoras para os alunos

1. O que acontece quando há muitas ovelhas e poucos lobos no início?
2. O que acontece quando há muitos lobos e poucas ovelhas?
3. Aumentar o custo de movimento favorece qual população?
4. Aumentar a energia obtida da grama torna o sistema mais estável ou mais instável?
5. Aumentar a energia obtida pelos lobos ao comer ovelhas favorece coexistência ou extinção?
6. Em que condições as oscilações populacionais ficam mais evidentes?
7. A mesma configuração de parâmetros produz sempre o mesmo resultado quando a semente aleatória muda?
8. O que este modelo simplifica em relação a um ecossistema real?


## Apêndice técnico opcional — fallback sem painel

A V5 incluía uma célula de fallback que executava imediatamente uma simulação completa sem sliders e sem botões. Ela foi útil durante o diagnóstico, mas não faz parte do fluxo didático principal porque o painel SVG/HTML funcionou.

Para evitar que uma célula posterior interfira na experiência visual do painel principal, o fallback não é executado automaticamente nesta versão. Caso seja necessário recuperar essa rotina, use a célula 6 da V5 como referência técnica.
